### Lab 10 Meeting 2 ###

In [219]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Operator, Statevector
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from itertools import batched
import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import expm

In [127]:
def create_bell_pair():
    qc = QuantumCircuit(2,2)
    qc.h(0)
    qc.cx(0,1)
    return qc

In [89]:
def encode_message(qc, message):
    if message == "01":
        qc.x(0)
    elif message == "10":
        qc.z(0)
    elif message == "11":
        qc.z(0)
        qc.x(0)
    return qc    

In [90]:
def decode_message(qc):
    qc.cx(0,1)
    qc.h(0)
    return qc

In [192]:
def measure_circuit(qc):
    qc.measure(0,1)
    qc.measure(1,0)
    qc.draw()
    backend = AerSimulator()
    result = backend.run(transpile(qc, backend), shots =1000).result()
    first_key = next(iter(result.get_counts()))
    return(first_key)

In [211]:
def string_encode(message : str):
    binary_list = [format(ord(c), '08b') for c in message]
    binary_string = ''
    for b in binary_list:
        binary_string += b
    return binary_string


In [237]:
def string_decode(binary_list):
    string = ''
    # for binary_str in binary_list:
    for b1, b2, b3, b4, b5, b6, b7, b8 in batched(binary_list, 8):
        char = chr(int(str(b1) + str(b2) + str(b3) + str(b4) + str(b5) + str(b6) + str(b7) + str(b8), 2))
        string = string + char
    return(string)

In [232]:
def get_two_bit_array(binary_string):
    new_binary_string = []
    it = iter(binary_string)
    for x, y in batched(binary_string, 2):
        new_binary_string.append(str(x) + str(y))
    return new_binary_string

def get_bit_string(binary_list):
    new_binary_string = []
    temp_sting = ''
    itter = 0
    for b in binary_list:
        if(itter < 4):
            temp_sting += b
        else:
            new_binary_string.append(temp_sting)
            itter = 0
            temp_sting = b
    new_binary_string.append(temp_sting)
    return new_binary_string[0]

In [36]:
test = ['01001000',
        '01100001',
        '01110000',
        '01110000',
        '01111001',
        '00100000',
        '01101000',
        '01101111',
        '01101100',
        '01101001',
        '01100100',
        '01100001',
        '01111001',
        '01110011',
        '00100001']

In [ ]:
# Encode the string
bit_string = string_encode('hello')
# Format the bits into a two bit array
bit_array = get_two_bit_array(bit_string)
# Empty array for quantum circuits
qc_arr = []
# Get each 2-bits and make a quantum circuit
for bits in bit_array:
    qc = create_bell_pair()
    qc_arr.append(encode_message(qc, bits))

# Decode string
decode_string = ''
# For each circuit gotten, take a measurement and append the string
for circuit in qc_arr:
    decode_message(circuit)
    measure_circuit(circuit)
    backend = AerSimulator()
    result = backend.run(transpile(circuit, backend), shots =1000).result()
    first_key = next(iter(result.get_counts()))
    decode_string += first_key

# decode the message
recived_message = string_decode(decode_string)
print(recived_message)